## Day 2 - Part 3: 종합 실습 과제

지금까지 배운 모든 평가 및 검증 기법을 활용하여, `단순한 모델`과 `복잡한 모델`의 성능을 종합적으로 비교하고 최적의 모델을 선택하는 과제를 수행해 봅시다.

`과제 목표:`

1.  두 가지 다른 구조의 모델을 정의합니다.

2.  조기 종료와 모델 체크포인트를 포함한 훈련 루프를 사용하여 각 모델을 훈련시킵니다.
3.  학습 곡선을 시각화하여 각 모델의 훈련 과정을 분석합니다. (과적합, 수렴 속도 등)
4.  최종적으로 저장된 `최고의 모델`을 사용하여 테스트 세트에서 성능을 평가합니다. (혼동 행렬, 분류 리포트)
5.  모든 결과를 종합하여 어떤 모델이 왜 더 나은 선택인지 논리적으로 설명합니다.

### 1. 사전 준비: 라이브러리 임포트 및 데이터 준비

먼저 필요한 라이브러리를 임포트하고, Day 2-Part 3 튜토리얼에서와 동일한 방식으로 위스콘신 유방암 데이터셋을 준비합니다. (훈련/검증/테스트 분할 및 스케일링 포함)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report
import os

# 데이터 로드
X, y = load_breast_cancer(return_X_y=True)

# 훈련+검증 / 테스트 분리
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 훈련 / 검증 분리
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, random_state=42, stratify=y_train_val
)

# 데이터 스케일링
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

# TODO: BreastCancerDataset 클래스 구현
# 지시사항:
# 1. torch.utils.data.Dataset을 상속받는 BreastCancerDataset 클래스를 정의하세요
# 2. __init__ 메서드에서 features와 labels를 매개변수로 받으세요
# 3. features를 torch.FloatTensor로, labels를 torch.LongTensor로 변환하여 저장하세요
# 4. __len__ 메서드를 구현하여 데이터셋의 크기를 반환하세요
# 5. __getitem__ 메서드를 구현하여 인덱스에 해당하는 features와 labels를 반환하세요

class BreastCancerDataset(Dataset):
    def __init__(self, features, labels):
        # TODO: 여기에 코드를 작성하세요
        pass
    
    def __len__(self):
        # TODO: 여기에 코드를 작성하세요
        pass
    
    def __getitem__(self, idx):
        # TODO: 여기에 코드를 작성하세요
        pass

# TODO: DataLoader 생성
# 지시사항:
# 1. 위에서 정의한 BreastCancerDataset을 사용하여 train_dataset, val_dataset, test_dataset을 생성하세요
# 2. 각각에 대해 적절한 DataLoader를 생성하세요:
#    - train_loader: batch_size=32, shuffle=True
#    - val_loader: batch_size=32, shuffle=False
#    - test_loader: batch_size=len(test_dataset), shuffle=False

# TODO: 여기에 코드를 작성하세요

print("데이터 준비 완료!")

### 2. 모델 정의: Simple vs. Complex

두 가지 다른 복잡도를 가진 모델을 정의합니다.
- `SimpleModel`: 은닉층 1개를 가진 간단한 모델
- `ComplexModel`: 은닉층 3개와 더 많은 뉴런을 가져 과적합 경향이 있는 복잡한 모델

In [ ]:
# TODO: SimpleModel 클래스 구현
# 지시사항: 
# 1. nn.Module을 상속받는 SimpleModel 클래스를 정의하세요
# 2. __init__ 메서드에서 num_features와 num_classes를 매개변수로 받으세요
# 3. 은닉층 1개를 가진 간단한 신경망을 구현하세요:
#    - 입력층: num_features → 16 뉴런
#    - 활성화 함수: ReLU
#    - 출력층: 16 → num_classes 뉴런
# 4. forward 메서드를 구현하여 입력 x를 네트워크에 통과시키세요

class SimpleModel(nn.Module):
    def __init__(self, num_features, num_classes):
        # TODO: 여기에 코드를 작성하세요
        pass
    
    def forward(self, x):
        # TODO: 여기에 코드를 작성하세요
        pass

# TODO: ComplexModel 클래스 구현
# 지시사항:
# 1. nn.Module을 상속받는 ComplexModel 클래스를 정의하세요
# 2. __init__ 메서드에서 num_features와 num_classes를 매개변수로 받으세요
# 3. 은닉층 3개를 가진 복잡한 신경망을 구현하세요:
#    - 입력층: num_features → 256 뉴런
#    - 첫 번째 은닉층: 256 → 128 뉴런
#    - 두 번째 은닉층: 128 → 64 뉴런
#    - 출력층: 64 → num_classes 뉴런
#    - 각 은닉층 후에 ReLU 활성화 함수를 사용하세요
#    - 규제 기법(Dropout, BatchNorm)은 사용하지 마세요
# 4. forward 메서드를 구현하여 입력 x를 네트워크에 통과시키세요

class ComplexModel(nn.Module):
    def __init__(self, num_features, num_classes):
        # TODO: 여기에 코드를 작성하세요
        pass
    
    def forward(self, x):
        # TODO: 여기에 코드를 작성하세요
        pass

### 3. 훈련 함수 작성

실습자료에서 작성한 `train_with_early_stopping` 함수를 여기에 그대로 가져와 사용합니다. 이 함수는 조기 종료와 모델 체크포인트 기능을 모두 포함하고 있습니다.

In [ ]:
def train_with_early_stopping(model, model_name, train_loader, val_loader, epochs=200, patience=15):
    # TODO 1: 모델 인스턴스 생성
    model = AdvancedClassifier(num_features=X_train.shape[1], num_classes=2)

    # TODO 2: 모델 체크포인트 경로 설정
    model_path = f"models/{model_name}.pth"

    # TODO 3: 최적화 설정
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    # TODO 4: 손실 함수 설정
    criterion = nn.CrossEntropyLoss()

### 4. 모델 학습 및 학습 곡선 분석

`지시사항:`
1. `SimpleModel`과 `ComplexModel`의 인스턴스를 각각 생성하세요.
2. 위에서 정의한 `train_with_early_stopping` 함수를 사용하여 두 모델을 모두 훈련시키고, `history`를 각각 저장하세요.
3. 두 모델의 학습 곡선(훈련 손실, 검증 손실)을 하나의 그래프에 시각화하여 비교하세요.
4. Markdown 셀에 학습 곡선 그래프를 보고 분석한 내용을 작성하세요. (예: 어떤 모델이 과적합 경향을 보이는가? 그 이유는 무엇인가? 조기 종료는 각 모델에서 언제쯤 발생했는가?)

In [ ]:
# TODO 1: 모델 인스턴스 생성
# SimpleModel과 ComplexModel의 인스턴스를 각각 생성하세요
# 힌트: num_features는 X_train.shape[1]로, num_classes는 2로 설정하세요

# TODO 2: 두 모델 훈련
# train_with_early_stopping 함수를 사용하여 두 모델을 모두 훈련시키세요
# 각 모델의 best_model과 history를 저장하세요
# 힌트: simple_model과 complex_model을 각각 훈련시키고 결과를 저장하세요

# TODO 3: 학습 곡선 시각화
# 두 모델의 학습 곡선(훈련 손실, 검증 손실)을 하나의 그래프에 시각화하세요
# plotly를 사용하여 비교 가능한 그래프를 생성하세요
# 힌트: go.Figure()를 사용하고, 각 모델의 train_loss와 val_loss를 다른 색상으로 표시하세요

#### `TODO 4: 학습 곡선 분석 결과`

*(여기에 분석 내용을 작성하세요)*

* `과적합 경향`: ...
* `수렴 속도 및 안정성`: ...
* `조기 종료 시점`: ...

### 5. 최종 성능 평가 및 비교 분석

`지시사항:`
1. 각 모델의 `저장된 최고 성능 버전(`best_..._model`)`을 사용하여 `test_loader`의 데이터에 대한 예측을 수행하세요.
2. 각 모델의 예측 결과에 대한 혼동 행렬을 시각화하고, `classification_report`를 출력하세요.
3. 두 모델의 최종 성능 지표(특히 재현율과 F1-Score)를 비교하고, 어떤 모델이 이 유방암 진단 문제에 더 적합한지 결론을 내리세요. 그 이유를 논리적으로 설명해야 합니다.

In [ ]:
# 평가를 위한 함수
def evaluate_model(model, model_name, test_loader):
    print(f"\n--- {model_name} Final Evaluation ---")
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for features, labels in test_loader:
            outputs = model(features)
            _, predicted = torch.max(outputs.data, 1)
            all_preds.extend(predicted.numpy())
            all_labels.extend(labels.numpy())
            
    # 혼동 행렬 및 분류 리포트
    cm = confusion_matrix(all_labels, all_preds)
    class_names = ['Malignant(악성)', 'Benign(양성)']
    fig = px.imshow(cm, labels=dict(x="Predicted", y="Actual"), x=class_names, y=class_names, text_auto=True, title=f'{model_name} Confusion Matrix')
    fig.show()
    
    print(classification_report(all_labels, all_preds, target_names=class_names))

# TODO 1 & 2: 각 모델 평가


#### `TODO 3: 최종 결론`

*(여기에 최종 결론을 작성하세요)*

이 과제를 통해 나는 `SimpleModel`과 `ComplexModel`을 비교 분석했습니다. 최종적으로 `___Model`을 선택하겠습니다. 그 이유는 다음과 같습니다.

1.  `일반화 성능`: 학습 곡선 분석 결과, ...
2.  `성능 지표`: 테스트 세트 평가 결과, 특히 유방암 진단에서 중요한 '악성(Malignant)' 클래스에 대한 재현율(Recall)이 ...
3.  `모델의 효율성`: ...

따라서 ... 하다고 결론 내릴 수 있습니다.